In [9]:
from statemachine import State, StateMachine
import simpy
import functools
import pprint

def simpy_process(func):
  @functools.wraps(func)
  # translate fuction call to simpy generator process
  def wrapper_simpy_process(*args, **kwargs):
    args[0].env.process(func(*args, **kwargs))
  return wrapper_simpy_process

class TrafficLight(StateMachine):

  red = State(initial=True)
  green = State()
  yellow = State()
  cycle = green.to(yellow) | yellow.to(red) | red.to(green)

  def __init__(self, env, name):
    self.env = env
    self.name = name
    self.cycle_event = self.env.event()
    self.cycle_event.callbacks.append(self.translate_cycle)
    self.timeout = 0
    StateMachine.__init__(self)

  def translate_cycle(self, event):
    # translate simpy event to statemachine event
    print("Translating cycle....")
    self.cycle_event = self.env.event()
    self.cycle_event.callbacks.append(self.translate_cycle)
    self.cycle()

  @simpy_process
  def on_enter_red(self):
    self.timeout = self.env.timeout(30)
    ret = yield (self.timeout | self.cycle_event)
    print(f"{self.name} done RED: {self.env.now}")
    if ret == {self.timeout: None}:
      self.cycle_event.succeed()

  @simpy_process
  def on_enter_green(self):
    yield self.env.timeout(25)
    print(f"{self.name} done GREEN: {self.env.now}")
    self.cycle()
  
  @simpy_process
  def on_enter_yellow(self):
    yield self.env.timeout(5)
    print(f"{self.name} done YELLOW: {self.env.now}")
    self.cycle()

def main():
  env = simpy.Environment()
  t1 = TrafficLight(env, "Water St.")
  t2 = TrafficLight(env, "Main St.")
  env.run(until=200)

if __name__ == '__main__':
  main()

Water St. done RED: 30
Main St. done RED: 30
Translating cycle....
Translating cycle....
Water St. done GREEN: 55
Main St. done GREEN: 55
Water St. done YELLOW: 60
Main St. done YELLOW: 60
Water St. done RED: 90
Main St. done RED: 90
Translating cycle....
Translating cycle....
Water St. done GREEN: 115
Main St. done GREEN: 115
Water St. done YELLOW: 120
Main St. done YELLOW: 120
Water St. done RED: 150
Main St. done RED: 150
Translating cycle....
Translating cycle....
Water St. done GREEN: 175
Main St. done GREEN: 175
Water St. done YELLOW: 180
Main St. done YELLOW: 180


In [12]:
from statemachine import StateMachine
from param import Parameterized
from abc import ABCMeta


class StateMachineParam(Parameterized, StateMachine, metaclass=ABCMeta):
  """
  Metaclass for combining Parameterized and StateMachine behavior.
  """
  def __new__(mcs, name, bases, attrs):
    # Combine attributes from both bases
    combined_attrs = dict(StateMachine.__dict__)
    combined_attrs.update(Parameterized.__dict__)
    combined_attrs.update(attrs)
    return type.__new__(mcs, name, bases, combined_attrs)

class MyStateMachine(StateMachineParam):
  """
  Custom state machine with parameters.
  """
  states = ['idle', 'running', 'finished']
  initial_state = 'idle'

  # Define parameters using param library
  mode = Parameter(default='normal', doc="Execution mode (normal, fast)")

  def transition(self, event):
    if event == 'start':
      if self.mode == 'fast':
        self.go_to('running')
      else:
        self.go_to('slow_start')
    elif event == 'continue':
      if self.current_state == 'slow_start':
        self.go_to('running')
    elif event == 'stop':
      self.go_to('finished')
    else:
      raise MachineError(f"Invalid event: {event}")

  def execute(self):
    """
    Perform actions based on the current state.
    """
    if self.current_state == 'running':
      print("Executing in normal mode...")
    elif self.current_state == 'finished':
      print("Execution completed!")
    else:
      print("Waiting for start...")

# Example usage
machine = MyStateMachine(mode='fast')
machine.transition('start')
machine.execute()  # Output: Executing in normal mode... (if mode is not set to 'fast')
machine.transition('continue')
machine.execute()  # Output: Executing in normal mode...
machine.transition('stop')
machine.execute()  # Output: Execution completed!

TypeError: metaclass conflict: the metaclass of a derived class must be a (non-strict) subclass of the metaclasses of all its bases